# LocalStoryChat Writer Training
### Qwen3.5-9B QLoRA on Google Colab

This notebook is prepared for the free Google Colab T4 runtime. It downloads the LocalStoryChat training branch, rebuilds the 48-example Gold dataset, checks the GPU, preprocesses the Axolotl configuration, then runs the first writer-training experiment.

**Before running:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU**. Keep the notebook runtime ordinary and do not select a paid accelerator.

In [ ]:
import subprocess, sys
print('Python:', sys.version)
subprocess.run(['nvidia-smi'], check=False)


## 1. Clone the project
The repository contains the 48 reusable Gold examples and the training configuration.

In [ ]:
import os, subprocess
repo_dir = '/content/LocalStoryChat'
if not os.path.exists(repo_dir):
    subprocess.run(['git','clone','--branch','generic-story-engine-rebuild','--single-branch','https://github.com/tony21675/LocalStoryChat.git',repo_dir], check=True)
os.chdir(repo_dir)
print('Project:', os.getcwd())


## 2. Install Axolotl
This is the only substantial software-install cell.

In [ ]:
%pip install -q --no-build-isolation "axolotl>=0.16.1"


If Colab says the Python environment needs to restart after installation, use **Runtime → Restart session**, then start again at the next cell.

In [ ]:
import os, subprocess
os.chdir('/content/LocalStoryChat')
subprocess.run(['python','training/prepare_gold_dataset.py'], check=True)


In [ ]:
from pathlib import Path
gold = sorted(Path('training/gold').glob('*.json'))
dataset = Path('training/gold_dataset.jsonl')
print('Gold examples:', len(gold))
print('Dataset exists:', dataset.exists())
print('Dataset size:', dataset.stat().st_size if dataset.exists() else 0, 'bytes')
assert len(gold) == 48, 'Expected 48 Gold examples'
assert dataset.exists(), 'Gold dataset was not created'


## 3. Preprocess first
We do this before the actual training run so a dataset, template, or configuration problem is caught before consuming GPU time.

In [ ]:
import os, subprocess
os.chdir('/content/LocalStoryChat')
subprocess.run(['axolotl','preprocess','training/qwen3_5_9b_kaggle_t4.yaml'], check=True)


## 4. Train
The first pass is conservative: QLoRA 4-bit, sequence length 2048, micro-batch 1, gradient accumulation 8, three epochs, FP16, SDPA, and no sample packing.

In [ ]:
import os, subprocess
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.chdir('/content/LocalStoryChat')
subprocess.run(['axolotl','train','training/qwen3_5_9b_kaggle_t4.yaml'], check=True)


## 5. Check the trained adapter

In [ ]:
from pathlib import Path
out = Path('training/outputs/qwen3_5_9b_writer_kaggle')
print('Output exists:', out.exists())
if out.exists():
    for p in sorted(out.rglob('*')):
        if p.is_file():
            print(p)


## 6. Package the adapter
This creates a ZIP in `/content` so it is easy to download from the Colab file panel.

In [ ]:
import shutil
archive = shutil.make_archive('/content/qwen3_5_9b_writer_adapter', 'zip', 'training/outputs/qwen3_5_9b_writer_kaggle')
print('Created:', archive)
